# Day 10: Final Model Selection and Production-Candidate Training

**Project:** IPL Player Performance Analytics and Prediction Using Machine Learning and Deep Learning

## Objectives
1. Carry forward the model selected by Day 8 using validation MAE (validation RMSE as tie-breaker).
2. Recreate the established chronological 70% / 15% / 15% split.
3. Refit the selected model on the combined training + validation partitions.
4. Evaluate the resulting candidate once on the untouched test partition.
5. Save versioned candidate artifacts, a feature manifest, test predictions, and documentation.

**Important:** test metrics are for final reporting only. They must not be used to change the selected model. This is a production-candidate artifact, not a claim that the model is production-ready.

In [1]:
from pathlib import Path
import json
import warnings
import platform
from datetime import datetime, timezone

import numpy as np
import pandas as pd
import sklearn
import joblib
from IPython.display import display
from sklearn.dummy import DummyRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LinearRegression
from sklearn.pipeline import Pipeline
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

try:
    from xgboost import XGBRegressor
    XGBOOST_AVAILABLE = True
    XGBOOST_IMPORT_ERROR = ""
except Exception as exc:
    XGBOOST_AVAILABLE = False
    XGBOOST_IMPORT_ERROR = repr(exc)

warnings.filterwarnings("ignore", category=FutureWarning)
current_dir = Path.cwd().resolve()
candidates = [current_dir, *current_dir.parents]
def is_project_root(p):
    return (
        (p / "data" / "processed" / "batting_next_match_features.csv").is_file()
        or ((p / "data" / "processed").is_dir() and (p / "notebooks").is_dir())
    )
PROJECT_ROOT = next((p for p in candidates if is_project_root(p)), None)
if PROJECT_ROOT is None:
    PROJECT_ROOT = Path(r"D:\Git\Personal\ipl-player-performance")

PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
DAY8_REPORT_DIR = PROJECT_ROOT / "reports" / "day8"
DAY10_REPORT_DIR = PROJECT_ROOT / "reports" / "day10"
MODEL_DIR = PROJECT_ROOT / "models" / "day10"
DOCS_DIR = PROJECT_ROOT / "docs"
for folder in [DAY10_REPORT_DIR, MODEL_DIR, DOCS_DIR]:
    folder.mkdir(parents=True, exist_ok=True)

BATTING_PATH = PROCESSED_DIR / "batting_next_match_features.csv"
BOWLING_PATH = PROCESSED_DIR / "bowling_next_match_features.csv"
SELECTION_PATH = DAY8_REPORT_DIR / "validation_model_selection.csv"
print("Project root:", PROJECT_ROOT)
for path in [BATTING_PATH, BOWLING_PATH, SELECTION_PATH]:
    print(("FOUND  " if path.is_file() else "MISSING"), path)
print("XGBoost available:", XGBOOST_AVAILABLE)
if not XGBOOST_AVAILABLE:
    print("XGBoost import issue:", XGBOOST_IMPORT_ERROR)

Project root: D:\Git\Personal\ipl-player-performance
FOUND   D:\Git\Personal\ipl-player-performance\data\processed\batting_next_match_features.csv
FOUND   D:\Git\Personal\ipl-player-performance\data\processed\bowling_next_match_features.csv
FOUND   D:\Git\Personal\ipl-player-performance\reports\day8\validation_model_selection.csv
XGBoost available: True


## 1. Load the Day 4 datasets and Day 8 model decisions

The notebook deliberately stops if the selection report is missing or does not contain one selected model for each target. It does not silently choose a replacement based on test results.

In [2]:
def read_dataset(path, required):
    if not path.is_file():
        raise FileNotFoundError(f"Missing {path}. Complete Days 4 and 8 first.")
    df = pd.read_csv(path)
    missing = set(required) - set(df.columns)
    if missing:
        raise ValueError(f"{path.name} missing columns: {sorted(missing)}")
    df["date"] = pd.to_datetime(df["date"], errors="coerce")
    df["next_match_date"] = pd.to_datetime(df["next_match_date"], errors="coerce")
    return df

batting = read_dataset(
    BATTING_PATH,
    {"player_name", "match_id", "date", "next_match_date", "next_match_runs"}
)
bowling = read_dataset(
    BOWLING_PATH,
    {"player_name", "match_id", "date", "next_match_date",
     "next_match_wickets", "next_match_runs_conceded"}
)
if not SELECTION_PATH.is_file():
    raise FileNotFoundError(
        f"Missing {SELECTION_PATH}. Run Day 8 and ensure its selection report exists."
    )
selection_df = pd.read_csv(SELECTION_PATH)
required_selection_cols = {
    "dataset", "target", "selected_by_validation_mae", "validation_mae"
}
if not required_selection_cols.issubset(selection_df.columns):
    raise ValueError(
        "Day 8 selection report schema mismatch. Required: "
        f"{sorted(required_selection_cols)}; found: {list(selection_df.columns)}"
    )

print("Batting shape:", batting.shape)
print("Bowling shape:", bowling.shape)
display(selection_df)

Batting shape: (18030, 49)
Bowling shape: (14123, 50)


,dataset,target,selected_by_validation_mae,validation_mae,saved_pipeline
0,batting,next_match_runs,LinearRegression,15.864845,D:\Git\Personal\ipl-player-performance\models\...
1,bowling,next_match_wickets,LinearRegression,0.789086,D:\Git\Personal\ipl-player-performance\models\...
2,bowling,next_match_runs_conceded,XGBoost,8.490233,D:\Git\Personal\ipl-player-performance\models\...


## 2. Rebuild the eligible features and chronological partitions

Only numeric `prior_...` features are candidates, consistent with Days 7–9. This conservative rule does not replace a manual review of how each feature was constructed.

In [3]:
TARGETS = {
    "batting": ["next_match_runs"],
    "bowling": ["next_match_wickets", "next_match_runs_conceded"],
}
IDENTIFIERS = {
    "player_name", "player_id", "match_id", "next_match_id",
    "date", "next_match_date", "source_file"
}
CURRENT_OUTCOMES = {
    "runs", "balls_faced", "fours", "sixes", "strike_rate", "batting_position",
    "dismissed", "wickets", "runs_conceded", "legal_balls", "overs", "economy",
    "maidens", "dots", "wides", "noballs"
}

def get_prior_features(df):
    features = []
    for col in df.columns:
        if (
            col.startswith("prior_")
            and pd.api.types.is_numeric_dtype(df[col])
            and col not in IDENTIFIERS
            and col not in CURRENT_OUTCOMES
            and col not in TARGETS["batting"] + TARGETS["bowling"]
        ):
            features.append(col)
    return features

def split_chronologically(df, target):
    work = df.loc[df[target].notna() & df["date"].notna()].copy()
    work = work.sort_values(["date", "match_id", "player_name"], kind="stable").reset_index(drop=True)
    if len(work) < 20:
        raise ValueError(f"{target}: only {len(work)} usable rows; need at least 20 for this split.")
    n = len(work)
    train_end = int(n * 0.70)
    val_end = int(n * 0.85)
    if train_end < 2 or val_end <= train_end or val_end >= n:
        raise ValueError(f"{target}: cannot create non-empty chronological partitions.")
    return (
        work,
        work.iloc[:train_end].copy(),
        work.iloc[train_end:val_end].copy(),
        work.iloc[val_end:].copy()
    )

def make_model(model_name):
    if model_name == "DummyMedian":
        estimator = DummyRegressor(strategy="median")
    elif model_name == "LinearRegression":
        estimator = LinearRegression()
    elif model_name == "RandomForest":
        estimator = RandomForestRegressor(
            n_estimators=300, max_depth=None, min_samples_leaf=2,
            random_state=42, n_jobs=-1
        )
    elif model_name == "XGBoost":
        if not XGBOOST_AVAILABLE:
            raise ImportError("Day 8 selected XGBoost, but XGBoost cannot be imported in this environment.")
        estimator = XGBRegressor(
            n_estimators=300, max_depth=4, learning_rate=0.05,
            subsample=0.9, colsample_bytree=0.9,
            objective="reg:squarederror", random_state=42, n_jobs=-1
        )
    else:
        raise ValueError(f"Unsupported model name from Day 8: {model_name}")
    return Pipeline([("imputer", SimpleImputer(strategy="median")), ("model", estimator)])

def regression_metrics(actual, predicted):
    return {
        "MAE": float(mean_absolute_error(actual, predicted)),
        "RMSE": float(np.sqrt(mean_squared_error(actual, predicted))),
        "R2": float(r2_score(actual, predicted)) if len(actual) > 1 else np.nan,
        "mean_residual_actual_minus_predicted": float(np.mean(np.asarray(actual) - np.asarray(predicted)))
    }

tasks = [
    (batting, "batting", "next_match_runs"),
    (bowling, "bowling", "next_match_wickets"),
    (bowling, "bowling", "next_match_runs_conceded"),
]

## 3. Train the final candidate and evaluate the held-out test set

For each target, the model name comes from Day 8's validation-based selection report. After the decision is fixed, the candidate is fitted on train + validation rows and evaluated once on the later test rows. The test score is not used to select a different model.

In [4]:
metric_rows = []
prediction_frames = []
manifest_rows = []
split_rows = []
status_rows = []

for df, dataset_name, target in tasks:
    try:
        match = selection_df[
            (selection_df["dataset"].astype(str) == dataset_name)
            & (selection_df["target"].astype(str) == target)
        ]
        if len(match) != 1:
            raise ValueError(
                f"Expected exactly one Day 8 selection row for {dataset_name}/{target}; found {len(match)}."
            )
        selected_model = str(match.iloc[0]["selected_by_validation_mae"])
        work, train, validation, test = split_chronologically(df, target)
        feature_candidates = get_prior_features(df)
        usable_features = [
            col for col in feature_candidates
            if pd.to_numeric(train[col], errors="coerce").notna().any()
        ]
        if not usable_features:
            raise ValueError(f"No usable numeric prior-history features for {dataset_name}/{target}.")

        # Convert feature values numerically; unparseable values become missing and are imputed in the pipeline.
        X_trainval = work.loc[work.index.isin(pd.concat([train, validation]).index), usable_features].apply(
            pd.to_numeric, errors="coerce"
        )
        y_trainval = pd.to_numeric(
            pd.concat([train, validation])[target], errors="coerce"
        )
        X_test = test[usable_features].apply(pd.to_numeric, errors="coerce")
        y_test = pd.to_numeric(test[target], errors="coerce")
        if y_trainval.isna().any() or y_test.isna().any():
            raise ValueError(f"Unexpected missing target after partitioning for {target}.")

        final_pipeline = make_model(selected_model)
        final_pipeline.fit(X_trainval, y_trainval)
        test_pred = final_pipeline.predict(X_test)
        metrics = regression_metrics(y_test, test_pred)
        metric_rows.append({
            "dataset": dataset_name,
            "target": target,
            "selected_model_from_day8": selected_model,
            "features": len(usable_features),
            "train_rows": len(train),
            "validation_rows": len(validation),
            "final_fit_rows_train_plus_validation": len(X_trainval),
            "test_rows": len(test),
            "test_start": str(test["date"].min().date()),
            "test_end": str(test["date"].max().date()),
            **metrics
        })

        safe_target = target.replace("/", "_")
        model_path = MODEL_DIR / f"{dataset_name}_{safe_target}_candidate.joblib"
        joblib.dump(final_pipeline, model_path)

        pred = test[["player_name", "match_id", "date", "next_match_date"]].copy()
        pred["dataset"] = dataset_name
        pred["target"] = target
        pred["selected_model"] = selected_model
        pred["actual"] = y_test.to_numpy()
        pred["predicted"] = test_pred
        pred["residual_actual_minus_predicted"] = pred["actual"] - pred["predicted"]
        prediction_frames.append(pred)

        manifest_rows.append({
            "dataset": dataset_name,
            "target": target,
            "model_name": selected_model,
            "model_path": str(model_path),
            "feature_columns_json": json.dumps(usable_features),
            "training_rows_including_validation": int(len(X_trainval)),
            "training_start_date": str(pd.concat([train, validation])["date"].min().date()),
            "training_end_date": str(pd.concat([train, validation])["date"].max().date()),
            "test_start_date": str(test["date"].min().date()),
            "test_end_date": str(test["date"].max().date()),
            "created_utc": datetime.now(timezone.utc).isoformat(),
            "python_version": platform.python_version(),
            "pandas_version": pd.__version__,
            "scikit_learn_version": sklearn.__version__,
            "random_state": 42,
            "notes": "Candidate trained on chronological train+validation; test used only for final diagnostic."
        })
        split_rows.append({
            "dataset": dataset_name, "target": target,
            "all_usable_rows": len(work), "train_rows": len(train),
            "validation_rows": len(validation), "test_rows": len(test),
            "train_start": str(train["date"].min().date()),
            "train_end": str(train["date"].max().date()),
            "validation_start": str(validation["date"].min().date()),
            "validation_end": str(validation["date"].max().date()),
            "test_start": str(test["date"].min().date()),
            "test_end": str(test["date"].max().date())
        })
        status_rows.append({"dataset": dataset_name, "target": target, "status": "COMPLETED", "detail": ""})
        print(f"COMPLETED: {dataset_name}/{target} using {selected_model}")
    except Exception as exc:
        status_rows.append({
            "dataset": dataset_name, "target": target,
            "status": "FAILED", "detail": repr(exc)
        })
        print(f"FAILED: {dataset_name}/{target}: {exc}")

metrics_df = pd.DataFrame(metric_rows)
predictions_df = pd.concat(prediction_frames, ignore_index=True) if prediction_frames else pd.DataFrame()
manifest_df = pd.DataFrame(manifest_rows)
splits_df = pd.DataFrame(split_rows)
status_df = pd.DataFrame(status_rows)

display(status_df)
display(metrics_df)

COMPLETED: batting/next_match_runs using LinearRegression
COMPLETED: bowling/next_match_wickets using LinearRegression
COMPLETED: bowling/next_match_runs_conceded using XGBoost


,dataset,target,status,detail
0,batting,next_match_runs,COMPLETED,
1,bowling,next_match_wickets,COMPLETED,
2,bowling,next_match_runs_conceded,COMPLETED,


,dataset,target,selected_model_from_day8,features,train_rows,validation_rows,final_fit_rows_train_plus_validation,test_rows,test_start,test_end,MAE,RMSE,R2,mean_residual_actual_minus_predicted
0,batting,next_match_runs,LinearRegression,15,12621,2704,15325,2705,2024-04-13,2026-05-29,16.850733,22.619524,0.105080,2.555942
1,bowling,next_match_wickets,LinearRegression,15,9886,2118,12004,2119,2024-04-08,2026-05-29,0.767798,0.989258,0.026481,0.040727
2,bowling,next_match_runs_conceded,XGBoost,15,9886,2118,12004,2119,2024-04-08,2026-05-29,9.516070,12.015263,0.053118,3.337681


## 4. Export artifacts and the model manifest

The manifest records the exact feature names and model artifact path for each target. Keep it with the saved model files so the future Django API can apply the same feature order and preprocessing.

In [5]:
metrics_path = DAY10_REPORT_DIR / "final_candidate_test_metrics.csv"
predictions_path = DAY10_REPORT_DIR / "final_candidate_test_predictions.csv"
manifest_path = DAY10_REPORT_DIR / "model_manifest.csv"
splits_path = DAY10_REPORT_DIR / "final_split_summary.csv"
status_path = DAY10_REPORT_DIR / "task_status.csv"

metrics_df.to_csv(metrics_path, index=False)
predictions_df.to_csv(predictions_path, index=False)
manifest_df.to_csv(manifest_path, index=False)
splits_df.to_csv(splits_path, index=False)
status_df.to_csv(status_path, index=False)

print("Saved reports:")
for path in [metrics_path, predictions_path, manifest_path, splits_path, status_path]:
    print(path)
print("Saved model artifacts:")
for path in sorted(MODEL_DIR.glob("*_candidate.joblib")):
    print(path)

Saved reports:
D:\Git\Personal\ipl-player-performance\reports\day10\final_candidate_test_metrics.csv
D:\Git\Personal\ipl-player-performance\reports\day10\final_candidate_test_predictions.csv
D:\Git\Personal\ipl-player-performance\reports\day10\model_manifest.csv
D:\Git\Personal\ipl-player-performance\reports\day10\final_split_summary.csv
D:\Git\Personal\ipl-player-performance\reports\day10\task_status.csv
Saved model artifacts:
D:\Git\Personal\ipl-player-performance\models\day10\batting_next_match_runs_candidate.joblib
D:\Git\Personal\ipl-player-performance\models\day10\bowling_next_match_runs_conceded_candidate.joblib
D:\Git\Personal\ipl-player-performance\models\day10\bowling_next_match_wickets_candidate.joblib


## 5. Review and acceptance checklist

- [ ] Exactly three targets are listed: runs, wickets, and runs conceded.
- [ ] Every target status is `COMPLETED`.
- [ ] The selected model names match Day 8's `validation_model_selection.csv`.
- [ ] The saved candidate artifacts exist and their manifest lists the intended feature columns.
- [ ] Test metrics are recorded but were not used to change model selection.
- [ ] Review negative R², high MAE/RMSE, or unexpected results before presenting model quality claims.
- [ ] Record any issues in the project decision log.

**Known limitation:** the split is row-based, matching Days 7–9. Records from one match can straddle split boundaries. Before final academic reporting, consider a match-grouped or date-boundary split and compare whether the conclusions remain stable.

In [6]:
def as_text(frame):
    return frame.to_string(index=False) if frame is not None and not frame.empty else "No rows produced."

doc = f"""# Day 10: Final Model Selection and Production-Candidate Training

## Execution status
```text
{as_text(status_df)}
```

## Final candidate test metrics
```text
{as_text(metrics_df)}
```

## Model manifest
```text
{as_text(manifest_df)}
```

## Outputs
- `reports/day10/final_candidate_test_metrics.csv`
- `reports/day10/final_candidate_test_predictions.csv`
- `reports/day10/model_manifest.csv`
- `reports/day10/final_split_summary.csv`
- `reports/day10/task_status.csv`
- `models/day10/*_candidate.joblib`
- `docs/day10_final_model_selection.md`

## Method
- Model names are carried forward from Day 8 validation-based selection.
- Feature candidates are restricted to numeric `prior_...` columns.
- Chronological row split remains 70% training, 15% validation, 15% test.
- The selected model is refit on train + validation data.
- The held-out test partition is used only for final diagnostic metrics.
- Imputation remains inside the fitted scikit-learn pipeline.
- The manifest records feature columns, model path, dates, and library versions.
- No performance result is asserted until this notebook is executed successfully on the user's actual project data.

## Limitations
A row-based split can put records from the same match into different partitions. Candidate artifacts are not production-ready until feature generation, grouped/time-based validation, reproducibility, and API input validation are reviewed.

## Next
Day 11: establish a leakage-safe sequence dataset design for the LSTM/GRU experiments.
"""
doc_path = DOCS_DIR / "day10_final_model_selection.md"
doc_path.write_text(doc, encoding="utf-8")
print("Saved documentation:", doc_path)
print("Task status:")
display(status_df)

Saved documentation: D:\Git\Personal\ipl-player-performance\docs\day10_final_model_selection.md
Task status:


,dataset,target,status,detail
0,batting,next_match_runs,COMPLETED,
1,bowling,next_match_wickets,COMPLETED,
2,bowling,next_match_runs_conceded,COMPLETED,
